# MySQL Enterprise con KMIP

`./demo.sh` incluye esta etapa. Reutiliza `mysql/mysql-enterprise-0`; una instalación vacía usa MySQL Enterprise 8.3.0, con la variante `aarch64` en ARM64. Las credenciales del registro Oracle y la contraseña inicial están en `.env`. El acceso al registro debe estar habilitado previamente en la cuenta Oracle.

Vault sirve KMIP en `vault-kmip.vault.svc.cluster.local:5696`. MySQL carga `keyring_okv`; las tablas de Lumen usan `ENCRYPTION='Y'`. Los certificados de cliente existentes se conservan.


In [ ]:
%%bash
set -euo pipefail
bash scripts/demo/mysql.sh


## Comprobar plugin y cifrado

La consulta usa la contraseña del entorno del pod y no la muestra. Las tablas aparecerán después de ejecutar la etapa de roles de base de datos.


In [ ]:
%%bash
set -euo pipefail
kubectl --context workshop -n mysql exec -i mysql-enterprise-0 -c mysql -- \
  sh -c 'MYSQL_PWD="$MYSQL_ROOT_PASSWORD" exec mysql -uroot --batch' <<'SQL'
SELECT PLUGIN_NAME, PLUGIN_STATUS FROM INFORMATION_SCHEMA.PLUGINS WHERE PLUGIN_NAME='keyring_okv';
SELECT NAME, ENCRYPTION FROM INFORMATION_SCHEMA.INNODB_TABLESPACES WHERE NAME LIKE 'scripting/%';
SQL


La demo automática no cambia el puerto KMIP para simular fallos, no rota la master key de InnoDB y no reinstala funciones UDF. Eran ejercicios manuales del laboratorio previo; no son requisitos de Lumen. Tampoco reinicia MySQL en cada ejecución ni elimina sus PVC.
